# GHV2 Exploratory Data Analysis

Search & Rescue WiFi CSI system — bidirectional CSI from 4 ESP32 shouters placed at area corners.

**Run all cells.** If the CSV has no data yet, each section will display a "no data" state gracefully.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..') if not os.path.exists('csi_parser.py') else '.')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import eda_utils
import ghv2_viz

# ── Configuration ────────────────────────────────────────────────────────────────────────────────
CSV_PATH           = "data/processed/capture_6.0x4.0m_2026-03-15_120000.csv"   # update to your actual file
MANUAL_DIMS        = (None, None)   # override with (width_m, depth_m) if needed
CONFIDENCE_THRESHOLD = 0.70
PLOT_DPI           = 150

%matplotlib inline
plt.rcParams['figure.dpi'] = PLOT_DPI
print(f"Config: CSV={CSV_PATH}  dims={MANUAL_DIMS}  threshold={CONFIDENCE_THRESHOLD}")

## Section 1 — Data Loading & Schema

In [ ]:
try:
    df, area_dims = eda_utils.load_csv(CSV_PATH, manual_dims=MANUAL_DIMS or None)
    groups = eda_utils.group_columns(df)
    print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Area dimensions: {area_dims}")
    print(f"Column groups: { {k: len(v) for k, v in groups.items()} }")
    display(df[eda_utils.META_COLS].head())
except FileNotFoundError as e:
    print(f"[INFO] {e}\nCreate a CSV first using GlassHouseV2.py.")
    df, area_dims, groups = pd.DataFrame(), (None, None), {}

## Section 2 — Statistical Summary

In [ ]:
if len(df) == 0:
    print("[INFO] No data — skipping statistical summary.")
else:
    desc = eda_utils.describe_dataset(df, groups)
    print(f"Shape: {desc['shape']}")
    summary_rows = [
        {"group": k, "n_cols": v["n_cols"], "missing_pct": v["missing_pct"]}
        for k, v in desc.items() if isinstance(v, dict)
    ]
    display(pd.DataFrame(summary_rows))

    outliers = eda_utils.outlier_summary(df, groups)
    print("\nOutlier summary (IQR method):")
    display(pd.DataFrame(outliers).T)

In [ ]:
if len(df) > 0:
    # Show missing values for meta + scalar columns only (full 5133 is too wide)
    scalar_cols = groups.get("meta", [])
    for sid in [1, 2, 3, 4]:
        scalar_cols += [c for c in df.columns
                        if c in (f"s{sid}_rssi", f"s{sid}_noise_floor",
                                 f"s{sid}_tx_rssi", f"s{sid}_tx_noise_floor")]
    if scalar_cols:
        fig, ax = plt.subplots(figsize=(10, 3))
        sns.heatmap(df[scalar_cols].isna().T, ax=ax, cbar=False,
                    xticklabels=False, yticklabels=True, cmap="Reds")
        ax.set_title("Missing Values — Meta + Scalar Columns (red = missing)")
        plt.tight_layout()
        plt.show()

## Section 3 — Temporal Analysis

In [ ]:
if len(df) == 0:
    print("[INFO] No data — skipping temporal analysis.")
else:
    stats = eda_utils.temporal_stats(df)
    print(f"Sampling rate: {stats['sampling_rate_hz']} Hz  "
          f"(mean interval: {stats['mean_interval_ms']} ms ± {stats['std_interval_ms']} ms)")
    print(f"Gaps detected (> 400 ms): {stats['n_gaps']}")
    if stats["gap_list"]:
        print("Gap locations (timestamp_ms, duration_ms):")
        for ts, dur in stats["gap_list"][:10]:
            print(f"  t={ts}ms  gap={dur}ms")

    rssi_cols = [f"s{sid}_rssi" for sid in [1, 2, 3, 4] if f"s{sid}_rssi" in df.columns]
    if rssi_cols:
        fig, ax = plt.subplots(figsize=(12, 4))
        ts_sorted = df.sort_values("timestamp_ms")
        for col in rssi_cols:
            ax.plot(ts_sorted["timestamp_ms"], ts_sorted[col],
                    alpha=0.7, linewidth=0.8, label=col)
        ax.set_xlabel("timestamp_ms")
        ax.set_ylabel("RSSI (dBm)")
        ax.set_title("RSSI Time Series — All Shouters (listener-rx)")
        ax.legend()
        plt.tight_layout()
        plt.show()

## Section 4 — Spatial Analysis

In [ ]:
if len(df) == 0:
    print("[INFO] No data — skipping spatial analysis.")
else:
    cell_stats = eda_utils.per_cell_stats(df)
    print("Per-cell statistics:")
    display(cell_stats)

    fig, ax = plt.subplots(figsize=(8, 4))
    labels = [f"r{int(r)}c{int(c)}" for r, c in
              zip(cell_stats["grid_row"], cell_stats["grid_col"])]
    ax.bar(labels, cell_stats["count"], color="#4C9BE8")
    ax.set_xlabel("Grid cell")
    ax.set_ylabel("Sample count")
    ax.set_title("Sample Count per Grid Cell")
    plt.tight_layout()
    plt.show()

## Section 5 — Feature Analysis

In [ ]:
if len(df) == 0:
    print("[INFO] No data — skipping feature analysis.")
else:
    for sid in [1, 2, 3, 4]:
        amp_cols = [c for c in groups.get(f"s{sid}", []) if "_amp_" in c][:16]
        if not amp_cols:
            continue
        fig, ax = plt.subplots(figsize=(12, 3))
        df[amp_cols].plot.box(ax=ax, rot=90, showfliers=False)
        ax.set_title(f"Shouter {sid} — Listener-RX Amplitude (first 16 subcarriers)")
        ax.set_ylabel("Amplitude")
        plt.tight_layout()
        plt.show()

In [ ]:
if len(df) > 0:
    all_scalar = []
    for sid in [1, 2, 3, 4]:
        for grp_key in [f"s{sid}", f"s{sid}_tx"]:
            all_scalar += [c for c in groups.get(grp_key, [])
                           if c.endswith("_rssi") or c.endswith("_noise_floor")]
    if all_scalar:
        corr = df[all_scalar].corr()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr, ax=ax, annot=True, fmt=".2f", cmap="RdBu_r",
                    center=0, square=True, linewidths=0.5)
        ax.set_title("Correlation — Scalar Features (RSSI + Noise Floor)")
        plt.tight_layout()
        plt.show()

In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4),
                              subplot_kw={"projection": "polar"})
    for i, sid in enumerate([1, 2, 3, 4]):
        phase_vals = eda_utils.phase_polar_data(df, groups.get(f"s{sid}", []))
        if len(phase_vals) > 0:
            axes[i].hist(phase_vals, bins=64, density=True, color="#4C9BE8", alpha=0.8)
        axes[i].set_title(f"S{sid} Phase", pad=10)
    plt.suptitle("Phase Distribution — Listener-RX (polar histogram)")
    plt.tight_layout()
    plt.show()

## Section 6 — 3×3 Spatial Heatmap

Grid cell dimensions:
- Area dimensions parsed from filename (or MANUAL_DIMS override)
- `cell_width = total_width / 3`, `cell_depth = total_depth / 3`

In [ ]:
if area_dims != (None, None):
    w, d = area_dims
    cw, cd = w / 3.0, d / 3.0
    print(f"Area: {w:.1f} m × {d:.1f} m")
    print(f"Cell: {cw:.2f} m × {cd:.2f} m  (width × depth)")
else:
    print("Area dimensions unknown — using cell index labels. "
          "Set MANUAL_DIMS = (width_m, depth_m) in Config to add physical labels.")

In [ ]:
if len(df) == 0:
    print("[INFO] No data — skipping heatmap.")
else:
    cell_stats = eda_utils.per_cell_stats(df)
    grid_rssi = np.full((3, 3), np.nan)
    for _, row in cell_stats.iterrows():
        grid_rssi[int(row["grid_row"]), int(row["grid_col"])] = row["mean_rssi"]

    fig = ghv2_viz.render_heatmap(
        grid_rssi, area_dims,
        title="Mean RSSI per Cell (dBm) — Listener-RX Average",
        mode="raw",
    )
    plt.show()

In [ ]:
if len(df) > 0:
    cell_stats = eda_utils.per_cell_stats(df)   # re-compute; avoids cross-cell dependency
    grid_count = np.zeros((3, 3), dtype=float)
    for _, row in cell_stats.iterrows():
        grid_count[int(row["grid_row"]), int(row["grid_col"])] = row["count"]

    fig = ghv2_viz.render_heatmap(
        grid_count, area_dims,
        title="Sample Count per Cell",
        mode="raw",
        cmap_raw="Blues",
    )
    plt.show()

## Section 7 — Pairwise Relationships (Scalar Features)

In [ ]:
if len(df) == 0:
    print("[INFO] No data — skipping pairwise plot.")
else:
    scalar_cols = [f"s{sid}_rssi" for sid in [1,2,3,4] if f"s{sid}_rssi" in df.columns]
    scalar_cols += [f"s{sid}_noise_floor" for sid in [1,2,3,4]
                    if f"s{sid}_noise_floor" in df.columns]
    if len(scalar_cols) >= 2:
        pd.plotting.scatter_matrix(df[scalar_cols], figsize=(10, 10),
                                   alpha=0.3, diagonal='kde')
        plt.suptitle("Pairwise Scatter — RSSI + Noise Floor (all shouters)")
        plt.tight_layout()
        plt.show()

## Section 8 — Model & Labeling Recommendations

In [ ]:
print("=" * 60)
print("MODEL SELECTION GUIDANCE")
print("=" * 60)
print(eda_utils.model_recommendation(df))
print()
print("=" * 60)
print(eda_utils.labeling_recommendation())
print("=" * 60)